# RQ1 — Current Data Science Methodologies, Practices, and Tools

**RQ1:** *What are the current data science methodologies, practices, and tools adopted in the industry?*

**Analyses:**
1. CRISP-DM phase adoption frequency — descriptive statistics per phase
2. Kruskal-Wallis H test: frequency differences across practitioner roles
3. Dunn's post-hoc test with Bonferroni correction for the significant phase (Data Understanding)


In [ ]:
!pip install numpy pandas scikit-learn scikit_posthocs


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import scikit_posthocs as sp   # pip install scikit-posthocs
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from analysis_utils import load_data

warnings.filterwarnings('ignore')

df = load_data()
N  = len(df)
print(f"Dataset loaded: {N} respondents, {len(df.columns)} columns")
display(df.head())

Dataset loaded: 112 respondents, 452 columns


,source,respondent_id,collector_id,start_date,end_date,ip_address,email_address,language,is_consent_age_18_Response,country_Response,...,method_unit_testing_curr,method_unit_testing_plan,method_unit_testing_past,method_use_case_curr,method_use_case_plan,method_use_case_past,method_containerization_curr,method_containerization_plan,method_containerization_past,adoption_score
0,Thailand,114600975755,431174607,2024-05-15 18:09:34,2024-05-15 19:52:57,27.55.81.82,NaN,th,ยืนยัน,Thailand,...,1,0,0,1,0,0,0,1,0,11
1,Thailand,114600955316,431174607,2024-05-15 17:31:40,2024-05-15 17:34:40,49.127.67.150,NaN,th,ยืนยัน,Australia,...,0,0,0,0,0,0,0,0,0,0
2,Thailand,114600751960,431174607,2024-05-15 09:38:44,2024-05-15 09:43:54,27.55.75.162,NaN,th,ยืนยัน,Thailand,...,1,0,0,1,0,0,1,0,0,12
3,Thailand,114599930108,431174607,2024-05-14 14:37:05,2024-05-14 16:11:21,110.164.162.162,NaN,en,ยืนยัน,Thailand,...,0,0,0,0,0,0,0,0,0,0
4,Thailand,114599009507,431174607,2024-05-13 17:28:42,2024-05-13 17:34:24,49.229.236.124,NaN,th,ยืนยัน,Thailand,...,0,1,0,0,1,0,0,1,0,0


## CRISP-DM Phase Columns

In [ ]:
PHASE_COLS = {
    'phase_frequency_header_freq_business_understanding': 'Business Understanding',
    'freq_data_understanding':  'Data Understanding',
    'freq_data_preparation':    'Data Preparation',
    'freq_modeling':            'Modeling',
    'freq_evaluation':          'Evaluation',
    'freq_deployment':          'Deployment',
}

## 1. Descriptive Statistics per Phase

In [ ]:
rows = []
for col, label in PHASE_COLS.items():
    if col not in df.columns:
        print(f"WARNING: column '{col}' not found — skipping")
        continue
    vals = df[col].dropna()
    rows.append({
        'Phase':  label,
        'n':      len(vals),
        'Mean':   round(vals.mean(), 3),
        'Median': vals.median(),
        'SD':     round(vals.std(), 3),
    })

desc_df = pd.DataFrame(rows)
display(desc_df)

,Phase,n,Mean,Median,SD
0,Business Understanding,87,3.701,4.0,1.286
1,Data Understanding,86,3.977,4.0,1.137
2,Data Preparation,87,4.057,4.0,1.124
3,Modeling,87,3.253,3.0,1.391
4,Evaluation,86,3.558,4.0,1.252
5,Deployment,87,2.828,3.0,1.480


## 2. Kruskal-Wallis H Test: Phase Frequency ~ Role

In [ ]:
kw_rows = []
for col, label in PHASE_COLS.items():
    if col not in df.columns:
        continue
    groups = [
        grp[col].dropna().values
        for _, grp in df.groupby('role')
        if len(grp[col].dropna()) >= 3
    ]
    if len(groups) < 2:
        continue
    h, p = stats.kruskal(*groups)
    kw_rows.append({
        'Phase':       label,
        'H-Statistic': round(h, 3),
        'p-value':     round(p, 4),
        'Sig. (p<.05)': '*' if p < 0.05 else '',
    })

kw_df = pd.DataFrame(kw_rows)
display(kw_df)

,Phase,H-Statistic,p-value,Sig. (p<.05)
0,Business Understanding,6.671,0.3524,
1,Data Understanding,14.184,0.0277,*
2,Data Preparation,9.933,0.1275,
3,Modeling,3.911,0.6887,
4,Evaluation,1.415,0.9650,
5,Deployment,4.970,0.5477,


## 3. Dunn's Post-Hoc Test (Bonferroni) — Data Understanding

Only Data Understanding reached significance (H=17.254, p=0.008).  
Adjusted threshold: α' = 0.05 / 21 pairs ≈ 0.0024


In [ ]:
DU_COL = 'freq_data_understanding'
tmp = df[['role', DU_COL]].dropna()

# Per-role descriptives
desc_role = tmp.groupby('role')[DU_COL].agg(
    n='count', Median='median', Mean='mean', SD='std'
).round(2).reset_index()
print("Per-role descriptive statistics:")
display(desc_role)

Per-role descriptive statistics:


,role,n,Median,Mean,SD
0,Data Analyst,19,4.0,4.11,1.15
1,Data Engineer,14,4.0,3.64,0.63
2,Data Scientist,14,5.0,4.36,1.08
3,ML Engineer,6,4.0,3.83,1.17
4,Others,12,5.0,4.33,1.50
5,Programmer/SW Dev.,14,4.0,3.64,1.22
6,Project Manager,7,4.0,3.71,1.11


In [ ]:
# Dunn's test matrix
dunn_mat = sp.posthoc_dunn(tmp, val_col=DU_COL, group_col='role', p_adjust='bonferroni')
print("Bonferroni-corrected p-value matrix:")
display(dunn_mat.round(4))

Bonferroni-corrected p-value matrix:


,Data Analyst,Data Engineer,Data Scientist,ML Engineer,Others,Programmer/SW Dev.,Project Manager
Data Analyst,1.0000,0.9515,1.0000,1.0,1.0000,1.0000,1.0
Data Engineer,0.9515,1.0000,0.1471,1.0,0.0647,1.0000,1.0
Data Scientist,1.0000,0.1471,1.0000,1.0,1.0000,0.9276,1.0
ML Engineer,1.0000,1.0000,1.0000,1.0,1.0000,1.0000,1.0
Others,1.0000,0.0647,1.0000,1.0,1.0000,0.4479,1.0
Programmer/SW Dev.,1.0000,1.0000,0.9276,1.0,0.4479,1.0000,1.0
Project Manager,1.0000,1.0000,1.0000,1.0,1.0000,1.0000,1.0


In [ ]:
# Extract upper triangle; reverse-engineer raw p from Bonferroni
roles  = list(dunn_mat.columns)
pairs  = []
for i in range(len(roles)):
    for j in range(i + 1, len(roles)):
        ra, rb  = roles[i], roles[j]
        p_bonf  = dunn_mat.loc[ra, rb]
        p_raw   = p_bonf / 21
        pairs.append({'Role A': ra, 'Role B': rb,
                      'p_raw': round(p_raw, 4), 'p_Bonf': round(p_bonf, 4)})

pairs_df = pd.DataFrame(pairs).sort_values('p_raw')
# Show only pairs with raw p < 0.10
display(pairs_df[pairs_df['p_raw'] < 0.10].reset_index(drop=True))

,Role A,Role B,p_raw,p_Bonf
0,Data Engineer,Others,0.0031,0.0647
1,Data Engineer,Data Scientist,0.0070,0.1471
2,Others,Programmer/SW Dev.,0.0213,0.4479
3,Data Scientist,Programmer/SW Dev.,0.0442,0.9276
4,Data Analyst,Data Engineer,0.0453,0.9515
5,ML Engineer,Project Manager,0.0476,1.0000
6,ML Engineer,Programmer/SW Dev.,0.0476,1.0000
7,ML Engineer,Others,0.0476,1.0000
8,Data Scientist,Project Manager,0.0476,1.0000
9,Data Scientist,Others,0.0476,1.0000


## 4. Tool Usage Diverging Bar Chart (Figure 4)

Diverging bar chart showing tools sorted by *currently using* count.
Left side = plan to use (future); right side = currently using.
Past use shown as an additional left-extending segment.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from analysis_utils import load_data, CURRENT_TH, PAST_TH, PLAN_TH

df = load_data()

# -----------------------------------------------------------------------
# Tool column definitions  (col_name → display label)
# -----------------------------------------------------------------------
TOOL_COLS = {
    'tools_usage_header_tool_airflow': 'Airflow',
    'tool_databricks':                 'Databricks',
    'tool_dataflow':                   'Dataflow',
    'tool_jupyter_notebook':           'Jupyter Notebook',
    'tool_streamlit':                  'Streamlit',
    'tool_tensorflow_serving':         'TensorFlow Serving',
    'tool_torchserve':                 'TorchServe',
    'tool_coltjob':                    'ColtJob',
    'tool_python_packages':            'Python packages',
    'tool_bigquery':                   'BigQuery',
    'tool_blacklink':                  'Backlink',
    'tool_cdsw':                       'CDSW',
    'tool_dbt':                        'dbt',
    'tool_google_colab':               'Google Colab',
    'tool_apache_hive':                'Apache Hive',
    'tool_isource':                    'iSource',
    'tool_impala':                     'Impala',
    'tool_jupyter_lab':                'JupyterLab',
    'tool_mlflow':                     'MLflow',
    'tool_rapidminer':                 'RapidMiner',
    'tool_sagemaker':                  'SageMaker',
    'tool_tableau':                    'Tableau',
    'tool_r_packages':                 'R packages',
    'tool_sas':                        'SAS',
    'tool_scala':                      'Scala',
    'tool_sql':                        'SQL',
    'tool_tkinter':                    'Tkinter',
    'tool_pycharm':                    'PyCharm',
}

# -----------------------------------------------------------------------
# Count responses
# -----------------------------------------------------------------------
records = []
for col, label in TOOL_COLS.items():
    if col not in df.columns:
        continue
    curr = int((df[col] == CURRENT_TH).sum())
    plan = int((df[col] == PLAN_TH).sum())
    past = int((df[col] == PAST_TH).sum())
    records.append({'tool': label, 'curr': curr, 'plan': plan, 'past': past})

import pandas as pd
tdf = pd.DataFrame(records).sort_values('curr', ascending=True).reset_index(drop=True)

labels = tdf['tool'].tolist()
curr   = tdf['curr'].values
plan   = tdf['plan'].values
past   = tdf['past'].values

# -----------------------------------------------------------------------
# Diverging bar chart
# Left  side: plan-to-use (orange) + past-use (grey), both as negatives
# Right side: currently using (steel blue)
# -----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 8))

y = np.arange(len(labels))
bar_h = 0.65

color_curr = '#2166AC'   # blue
color_plan = '#F4A442'   # orange
color_past = '#BBBBBB'   # light grey

# Currently using — right
ax.barh(y, curr, height=bar_h, color=color_curr, label='Currently using')

# Plan to use — left (shown as negative)
ax.barh(y, -plan, height=bar_h, color=color_plan, label='Plan to use')

# Past use — left of plan-to-use (stacked further left)
ax.barh(y, -past, height=bar_h, left=-plan, color=color_past, label='Previously used')

# -----------------------------------------------------------------------
# Axis formatting
# -----------------------------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=14)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Number of respondents', fontsize=14)

# Symmetric x-axis
max_val = max(curr.max(), (plan + past).max()) + 5
ax.set_xlim(-max_val, max_val)

# Positive tick labels on both sides
from matplotlib.ticker import FuncFormatter
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: str(abs(int(x)))))
ax.tick_params(axis='x', labelsize=14)

# Side labels
ax.text(-max_val + 1, len(labels) + 0.5, '← Plan to use / Previously used',
        ha='left', va='top', fontsize=14, color='grey')
ax.text(max_val - 1, len(labels) + 0.5, 'Currently using →',
        ha='right', va='top', fontsize=14, color='grey')

ax.legend(loc='lower right', fontsize=14, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'tool-usage-diverging.pdf')
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.savefig(out_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
print(f'Saved to {out_path}')
plt.show()

# Print summary table
print("\nTool usage summary (sorted by currently using):")
print(tdf.sort_values('curr', ascending=False).to_string(index=False))

Saved to /Users/chaiyong/Downloads/2023_SE4DS/EMSE/tool-usage-diverging.pdf

Tool usage summary (sorted by currently using):
              tool  curr  plan  past
   Python packages    62     8     9
               SQL    52    10    12
  Jupyter Notebook    44    12    19
        JupyterLab    39    16     9
      Google Colab    37    16    13
           Tableau    31    14    18
           Airflow    29    20    21
           PyCharm    26    19    16
          BigQuery    26    22    14
          Dataflow    26    22    12
        Databricks    24    26    17
       Apache Hive    24    28    16
            MLflow    23    23    15
TensorFlow Serving    23    33     9
               dbt    22    33     8
         Streamlit    20    30    14
        R packages    19    16    25
             Scala    17    23    18
         SageMaker    15    32    12
              CDSW    15    33    12
           Tkinter    15    21    18
               SAS    14    24    19
        TorchServe    14

## 5. CRISP-DM Phase Frequency Violin Plot (Figure 5)

Overall distribution of phase frequencies across all respondents, sorted by mean (descending).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from analysis_utils import load_data

import warnings; warnings.filterwarnings('ignore')
df = load_data()

# Phase columns ordered by mean frequency (descending)
PHASE_COLS_ORDERED = [
    ('freq_data_preparation',                                    'Data\nPreparation'),
    ('freq_data_understanding',                                  'Data\nUnderstanding'),
    ('phase_frequency_header_freq_business_understanding',       'Business\nUnderstanding'),
    ('freq_evaluation',                                          'Evaluation'),
    ('freq_modeling',                                            'Modeling'),
    ('freq_deployment',                                          'Deployment'),
]

data   = []
labels = []
for col, label in PHASE_COLS_ORDERED:
    if col in df.columns:
        vals = df[col].dropna().values
        data.append(vals)
        labels.append(label)

fig, ax = plt.subplots(figsize=(10, 5))

parts = ax.violinplot(data, positions=range(len(data)),
                      showmedians=True, showextrema=True)

# Style
for pc in parts['bodies']:
    pc.set_facecolor('#2166AC')
    pc.set_alpha(0.6)
parts['cmedians'].set_color('#D73027')
parts['cmedians'].set_linewidth(2)
parts['cmins'].set_color('#555555')
parts['cmaxes'].set_color('#555555')
parts['cbars'].set_color('#555555')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=14)
ax.set_yticks([0, 1, 2, 3, 4, 5])
ax.set_yticklabels(['0', '1', '2', '3', '4', '5'], fontsize=14)
ax.set_ylabel('Frequency (0 = Never, 5 = Very Frequently)', fontsize=14)
ax.set_ylim(-0.3, 5.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'phase_frequencies_violin.pdf')
plt.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Saved to {out_path}')
plt.show()

Saved to /Users/chaiyong/Downloads/2023_SE4DS/EMSE/phase_frequencies_violin.pdf


## 6. CRISP-DM Phase Frequency Correlation Heatmap (Figure 6)

Spearman correlation matrix across all six phases, using the same YlGnBu theme as Figure 3.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from analysis_utils import load_data
import warnings; warnings.filterwarnings('ignore')

df = load_data()

PHASE_COLS = {
    'phase_frequency_header_freq_business_understanding': 'BU',
    'freq_data_understanding':  'DU',
    'freq_data_preparation':    'DP',
    'freq_modeling':            'MOD',
    'freq_evaluation':          'EVA',
    'freq_deployment':          'DEP',
}

cols   = [c for c in PHASE_COLS if c in df.columns]
labels = [PHASE_COLS[c] for c in cols]
sub    = df[cols].rename(columns=dict(zip(cols, labels)))
corr   = sub.corr(method='spearman')

print('Spearman correlation matrix:')
display(corr.round(2))

plt.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='YlGnBu',
    vmin=0, vmax=1,
    cbar_kws={'label': 'Spearman correlation'},
    annot_kws={'size': 14},
    linewidths=0,
    ax=ax,
)
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=14)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=14)
fig.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'phase-freq-correlation.pdf')
fig.savefig(out_path, bbox_inches='tight')
print(f'Saved to {out_path}')
plt.show()

Spearman correlation matrix:


,BU,DU,DP,MOD,EVA,DEP
BU,1.00,0.55,0.18,0.07,0.19,0.16
DU,0.55,1.00,0.48,0.17,0.33,0.07
DP,0.18,0.48,1.00,0.19,0.15,0.12
MOD,0.07,0.17,0.19,1.00,0.46,0.20
EVA,0.19,0.33,0.15,0.46,1.00,0.26
DEP,0.16,0.07,0.12,0.20,0.26,1.00


Saved to /Users/chaiyong/Downloads/2023_SE4DS/EMSE/phase-freq-correlation.pdf


## 7. CRISP-DM Phase Frequency by Role — Violin Grid (Figure 7)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from analysis_utils import load_data

df = load_data()

PHASE_COLS = {
    'phase_frequency_header_freq_business_understanding': 'BU',
    'freq_data_understanding': 'DU',
    'freq_data_preparation':   'DP',
    'freq_modeling':           'MOD',
    'freq_evaluation':         'EVA',
    'freq_deployment':         'DEP',
}

ROLES = [
    'Data Analyst', 'Data Engineer', 'Data Scientist',
    'ML Engineer',  'Others',        'Programmer/SW Dev.', 'Project Manager',
]

cols   = [c for c in PHASE_COLS if c in df.columns]
labels = [PHASE_COLS[c] for c in cols]

fig, axes = plt.subplots(4, 2, figsize=(9, 11), sharey=True)
axes = axes.flatten()

for ax_idx, role in enumerate(ROLES):
    ax  = axes[ax_idx]
    sub = df[df['role'] == role]

    data = [sub[c].dropna().values for c in cols]

    # Use violinplot when n>=4, otherwise fall back to a strip+median marker
    if all(len(d) >= 4 for d in data):
        parts = ax.violinplot(data, positions=range(len(data)),
                              showmedians=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#2166AC')
            pc.set_alpha(0.55)
        parts['cmedians'].set_color('#D73027')
        parts['cmedians'].set_linewidth(2)
        parts['cmins'].set_color('#888888')
        parts['cmaxes'].set_color('#888888')
        parts['cbars'].set_color('#888888')
    else:
        for i, d in enumerate(data):
            jitter = np.random.default_rng(i).uniform(-0.15, 0.15, len(d))
            ax.scatter(np.full(len(d), i) + jitter, d,
                       color='#2166AC', alpha=0.7, s=20, zorder=3)
            if len(d):
                ax.hlines(np.median(d), i - 0.2, i + 0.2,
                          colors='#D73027', linewidth=2, zorder=4)

    n = len(sub[cols[0]].dropna())
    ax.set_title(f'{role}\n(n={n})', fontsize=13, pad=4)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_yticks([0, 1, 2, 3, 4, 5])
    ax.set_yticklabels(['0','1','2','3','4','5'], fontsize=12)
    ax.set_ylim(-0.4, 5.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if ax_idx % 2 == 0:
        ax.set_ylabel('Frequency (0–5)', fontsize=12)

# Hide the spare 8th panel
axes[-1].set_visible(False)

fig.tight_layout(h_pad=3, w_pad=2)

out_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'phase_frequencies_by_role_violin.pdf')
fig.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Saved to {out_path}')
plt.show()

Saved to /Users/chaiyong/Downloads/2023_SE4DS/EMSE/phase_frequencies_by_role_violin.pdf
